# Agent Debugging

This notebook adds a debugging tutorial on top of the existing workflow. It focuses on trace logs, execution inspection, and failure debugging without changing the original notebooks or the main workflow architecture.

## Learning goals

- Understand why traces matter for agent debugging.
- Inspect node-level inputs and outputs.
- Compare a normal run with a weaker or abstained run.
- Use failure evidence to reason about next improvements.


## Concept explanation

We begin with the environment check so trace inspection and artifact generation happen in the expected uv environment. Reproducible debugging starts with a reproducible interpreter.


In [ ]:
import sys
print(sys.executable)


This setup cell imports the additive trace debugging helpers from `src/trace_debug.py` and reuses the existing workflow and evaluation code. The original workflow stays intact; this notebook only adds a better debugging lens.


In [ ]:
import pandas as pd


from pathlib import Path
import os
import sys

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'src').exists() and (PROJECT_ROOT.parent / 'src').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
print(PROJECT_ROOT)

from src.evaluator import extract_failure_cases, run_evaluation_suite
from src.ingestion import build_demo_index
from src.trace_debug import display_node_inputs, display_node_outputs, display_trace
from src.workflow import run_workflow

pd.set_option('display.max_colwidth', 140)
retriever = build_demo_index(persist=False)


Trace logs matter because agent behavior is multi-step. A final answer alone rarely tells you whether the problem came from retrieval, planning, tool use, verification, or fallback. A good trace makes each step inspectable.

This debugging notebook focuses on three levels:

- full trace overview
- node-specific inputs
- node-specific outputs

## Implementation


In [ ]:
happy_state = run_workflow('How many days are in the pilot window?', retriever=retriever)
abstain_state = run_workflow('Who is the current CEO of the company?', retriever=retriever)
{
    'happy_status': happy_state['final_status'],
    'abstain_status': abstain_state['final_status'],
    'happy_steps': len(happy_state['trace']),
    'abstain_steps': len(abstain_state['trace']),
}


The first debugging view is the full trace table. This version includes node names, serialized inputs, serialized outputs, timestamps, and step-to-step latency derived from the timestamps.


In [ ]:
debug_trace_frame = display_trace(happy_state['trace'], render=False)
debug_trace_frame


## Trace Performance Analysis

Trace debugging is not only about correctness. It is also about performance. Latency per node helps you see whether retrieval, tool execution, synthesis, or verification is becoming the bottleneck as the workflow grows. The table below focuses on the node, its measured latency, and the serialized inputs and outputs.

In [ ]:
performance_frame = display_trace(happy_state['trace'], render=False)[['node', 'latency', 'inputs', 'outputs']]
performance_frame

A node with higher latency is not automatically bad. The key question is whether that latency is expected for the work being done. For example, retrieval may legitimately cost more than normalization, while unusually slow tool or synthesis steps can signal a debugging target.

Once you know which node matters, you often want to zoom in. The next two cells inspect only the inputs and only the outputs for a chosen node.


In [ ]:
display_node_inputs(happy_state['trace'], 'retrieve_docs', render=False)


Node outputs are just as important as inputs because they show what the workflow actually produced, not what you expected it to produce.


In [ ]:
display_node_outputs(happy_state['trace'], 'retrieve_docs', render=False)


## Experiment

A debugging experiment is to compare a grounded answered run and an abstained run. The two traces often differ less than people expect; the critical difference is usually in classification, evidence coverage, or fallback behavior.


In [ ]:
comparison = pd.DataFrame(
    [
        {
            'query': happy_state['user_query'],
            'final_status': happy_state['final_status'],
            'coverage_score': happy_state['verification_result'].coverage_score,
            'unsupported_claims': len(happy_state['verification_result'].unsupported_claims),
        },
        {
            'query': abstain_state['user_query'],
            'final_status': abstain_state['final_status'],
            'coverage_score': abstain_state['verification_result'].coverage_score,
            'unsupported_claims': len(abstain_state['verification_result'].unsupported_claims),
        },
    ]
)
comparison


Debugging also benefits from repeated evaluation. This cell runs a small evaluation pass, extracts the failure rows, and gives you a dataset-level view of what is going wrong.


In [ ]:
results, summary = run_evaluation_suite(repeats=1, persist_outputs=True)
failures = extract_failure_cases(results)
failures[['system', 'question_id', 'question', 'failure_type', 'predicted_status']].head(10)


## Result analysis

The debugging views show two complementary truths: traces help explain one run deeply, while evaluation failures help explain the overall pattern. Together they make debugging much more systematic than reading final answers alone.


In [ ]:
analysis_frame = pd.DataFrame(
    [
        {'debugging_view': 'full_trace', 'use_case': 'understand the whole execution path'},
        {'debugging_view': 'node_inputs', 'use_case': 'inspect what information reached a node'},
        {'debugging_view': 'node_outputs', 'use_case': 'inspect what the node actually produced'},
        {'debugging_view': 'failure_table', 'use_case': 'spot recurring issues across many runs'},
    ]
)
analysis_frame


## Takeaways

- Traces turn hidden agent behavior into inspectable evidence.
- Node-specific inspection is useful when a single stage looks suspicious.
- Evaluation failures complement traces by showing repeated weak spots.
- This notebook adds debugging education without rewriting the existing repo.
